In [25]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import month
from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import StandardScaler
from pyspark.ml.feature import PCA
from pyspark.ml.regression import LinearRegression
from pyspark.sql.functions import col

In [19]:
spark = SparkSession.builder \
  .appName("Preparação de Dados - Modelo de Progressão")\
  .getOrCreate()

In [20]:
df_video = spark.read.parquet("/content/videos-comments-tratados.snappy.parquet")

In [21]:
df_video = df_video.withColumn("Month", month("Published At"))

In [22]:
indexer = StringIndexer(inputCol="Keyword", outputCol="KeywordIndex")
df_video = indexer.fit(df_video).transform(df_video)

In [26]:
df_video = df_video.withColumn("Year", col("Year").cast("int"))

In [27]:
assembler = VectorAssembler(
    inputCols=["Likes", "Views", "Year", "Month", "KeywordIndex"],
    outputCol="Features"
)
df_video = assembler.transform(df_video)

In [28]:
scaler = StandardScaler(inputCol="Features", outputCol="Features Normal", withStd=True, withMean=False)
df_video = scaler.fit(df_video).transform(df_video)

In [29]:
pca = PCA(k=1, inputCol="Features Normal", outputCol="Features PCA")
df_video = pca.fit(df_video).transform(df_video)

In [30]:
train_data, test_data = df_video.randomSplit([0.8,0.2], seed=42)

In [31]:
lr = LinearRegression(featuresCol="Features Normal", labelCol="Comments")
lr_model = lr.fit(train_data)

In [32]:
test_results = lr_model.evaluate(test_data)

In [33]:
df_video.write.parquet("videos-preparados-parquet")

In [34]:
spark.stop()